# 01 · Hiểu dữ liệu Takeout trước khi thiết kế bộ phân loại

**Mục tiêu:** tìm những thông tin thực sự có trong dữ liệu, kiểm tra chất lượng và tạo bằng chứng để thiết kế cách chấm/duyệt video nhạc. Chưa huấn luyện model, chưa chốt regex phân loại, chưa tải audio.

Câu hỏi xuyên suốt:
1. Có bao nhiêu **sự kiện xem**, **video duy nhất**, bản ghi quảng cáo, bài đăng và dữ liệu thiếu?
2. Tiêu đề và kênh cung cấp tín hiệu gì? Bao nhiêu video có tiêu đề ít thông tin hoặc hệ chữ khác nhau?
3. Những giả thuyết từ khóa nào phủ được dữ liệu, mâu thuẫn hoặc để lại khoảng trống?
4. Cần gán nhãn những mẫu nào để đo bỏ sót/nhận nhầm trước khi chọn trọng số?

**Định nghĩa mục tiêu:** nội dung chính là âm nhạc, gồm AMV, cover, OST, unofficial upload, remix, live biểu diễn và BGM độc lập. Video nói chuyện/game có BGM không thuộc mục tiêu. Không suy ra nhãn từ category hoặc từ khóa đơn lẻ.

Notebook chạy offline. Link chỉ được truy cập khi bạn bấm vào; dữ liệu được giữ trong máy.


In [ ]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from collections import Counter
from hashlib import sha256
import json
import re
import unicodedata
import html
import sys
import platform

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 18)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print({'python': platform.python_version(), 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__})


## 1. Cấu hình và nguồn dữ liệu

Mặc định tìm repo từ thư mục đang mở hoặc thư mục cha. `TAKEOUT_DIR` nhận folder đã giải nén; `WATCH_HISTORY_FILE` dùng khi có nhiều file trùng tên. Không tự gộp nhiều export vì chưa có quy tắc xác định sự kiện trùng giữa các export.

Seed cố định giúp tái lập mẫu. Thời điểm nguồn được lưu UTC, biểu đồ hành vi chuyển sang `LOCAL_TIMEZONE`. Cả ngày đầu và cuối có thể là ngày chưa đầy đủ.


In [ ]:
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / '.ai-devkit.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ thư mục repo Auralytica hoặc notebooks/.')
TAKEOUT_DIR = ROOT / 'artifacts' / 'YouTube and YouTube Music'
WATCH_HISTORY_FILE = None  # Ví dụ: Path('/path/to/history/watch-history.json')
LOCAL_TIMEZONE = 'Asia/Bangkok'
SEED = 42
RANDOM_SAMPLE_SIZE = 300
DISCOVERY_SAMPLE_PER_STRATUM = 25
MAX_PER_CHANNEL_PER_STRATUM = 2
MAX_PER_CHANNEL_DISCOVERY = 5

if WATCH_HISTORY_FILE is not None:
    source = Path(WATCH_HISTORY_FILE).expanduser()
else:
    if not TAKEOUT_DIR.is_dir():
        raise FileNotFoundError(f'Không tìm thấy folder: {TAKEOUT_DIR}')
    matches = sorted(TAKEOUT_DIR.rglob('watch-history.json'))
    if not matches:
        html_count = len(list(TAKEOUT_DIR.rglob('watch-history.html')))
        raise FileNotFoundError(f'Cần watch-history.json. Tìm thấy {html_count} file HTML; notebook này chưa parse HTML.')
    if len(matches) != 1:
        raise ValueError(f'Tìm thấy {len(matches)} file JSON. Đặt WATCH_HISTORY_FILE để chọn một file.')
    source = matches[0]
source_bytes = source.read_bytes()
source_hash = sha256(source_bytes).hexdigest()
LEGACY_OUTPUT_DIR = ROOT / 'artifacts' / 'notebook-runs' / '01_takeout_eda' / source_hash[:12]
EDA_VERSION = 'signals-v2'
OUTPUT_DIR = LEGACY_OUTPUT_DIR / EDA_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
raw = json.loads(source_bytes)
if not isinstance(raw, list) or not all(isinstance(x, dict) for x in raw):
    raise ValueError('Đầu vào phải là JSON array gồm các object lịch sử.')
if not raw:
    raise ValueError('File lịch sử rỗng; không đủ dữ liệu để EDA.')
print({'source': str(source.relative_to(ROOT)) if source.is_relative_to(ROOT) else source.name,
       'sha256': source_hash, 'bytes': len(source_bytes), 'raw_events': len(raw)})


In [ ]:
# Chỉ liệt kê cấu trúc ở bước này, chưa suy diễn ngữ nghĩa từ tên trường.
field_counts = Counter(k for row in raw for k in row)
schema = pd.DataFrame([
    {'field': k, 'present': n, 'missing': len(raw)-n,
     'observed_types': ', '.join(sorted({type(row[k]).__name__ for row in raw if k in row}))}
    for k, n in field_counts.items()
]).sort_values('present', ascending=False)
display(schema)
print('Không có duration/watch-time/category trong file này:',
      all(k not in field_counts for k in ['duration', 'watchTime', 'categoryId']))


## 2. Chuẩn hóa sự kiện và kiểm tra chất lượng

Giữ `source_row`, tiêu đề và URL gốc để truy vết. Nhận URL `watch?v=`, `youtu.be`, `shorts`, `live`, `embed`; ID phải có 11 ký tự hợp lệ và host phải thuộc YouTube. URL không nhận diện được được giữ để khảo sát, không âm thầm bỏ đi.

`From Google Ads` là dấu hiệu quảng cáo quan sát được trong export hiện tại. Danh sách marker/prefix có thể cần mở rộng cho export ngôn ngữ khác; các nhóm chưa nhận diện sẽ được hiển thị. Sự kiện lặp nguyên bản được báo cáo, **không tự xóa** vì chưa biết đó là lỗi export hay hành vi thật.


In [ ]:
VIDEO_ID = re.compile(r'[A-Za-z0-9_-]{11}')
YOUTUBE_HOSTS = {'youtube.com', 'www.youtube.com', 'm.youtube.com', 'music.youtube.com'}
AD_MARKERS = {'from google ads'}

def classify_url(value):
    if not isinstance(value, str) or not value.strip():
        return 'missing_url', None
    try:
        u = urlparse(value)
        host = (u.hostname or '').lower()
        parts = u.path.strip('/').split('/')
        candidate = None
        if host in {'youtu.be', 'www.youtu.be'}:
            candidate = parts[0]
        elif host in YOUTUBE_HOSTS:
            if u.path.rstrip('/') == '/watch':
                candidate = parse_qs(u.query).get('v', [None])[0]
            elif parts[0] in {'shorts', 'live', 'embed'} and len(parts) > 1:
                candidate = parts[1]
            elif parts[0] == 'post':
                return 'community_post', None
            else:
                return 'other_youtube_url', None
        else:
            return 'other_host', None
        if candidate and VIDEO_ID.fullmatch(candidate):
            return 'video', candidate
        return 'invalid_video_url', None
    except ValueError:
        return 'malformed_url', None

def strings_in(items, key):
    return [item[key] for item in (items or [])
            if isinstance(item, dict) and isinstance(item.get(key), str)]

records = []
for i, row in enumerate(raw):
    kind, video_id = classify_url(row.get('titleUrl'))
    subtitles = row.get('subtitles') or []
    names = strings_in(subtitles, 'name')
    urls = strings_in(subtitles, 'url')
    details = strings_in(row.get('details'), 'name')
    raw_title = row.get('title') if isinstance(row.get('title'), str) else ''
    records.append({
        'source_row': i, 'event_kind': kind, 'video_id': video_id,
        'raw_title': raw_title,
        'title': re.sub(r'^(?:Watched|Đã xem)\s+', '', raw_title).strip(),
        'raw_url': row.get('titleUrl', ''), 'time_raw': row.get('time'),
        'channel': names[0] if names else '', 'channel_url': urls[0] if urls else '',
        'subtitle_count': len(names), 'details': ' | '.join(details),
        'is_ad': any(s.casefold().strip() in AD_MARKERS for s in details),
        'raw_record_hash': sha256(json.dumps(row, sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
    })
events = pd.DataFrame(records)
events['time_utc'] = pd.to_datetime(events['time_raw'], utc=True, errors='coerce', format='mixed')
events['time_local'] = events['time_utc'].dt.tz_convert(LOCAL_TIMEZONE)
display(pd.crosstab(events['event_kind'], events['is_ad']).rename(columns={False:'not_marked_ad', True:'marked_ad'}))
display(pd.DataFrame({
    'check': ['missing/invalid time', 'missing title', 'missing channel on video event',
              'multiple subtitles', 'exact duplicate raw records after first'],
    'events': [events.time_utc.isna().sum(), events.title.eq('').sum(),
               (events.event_kind.eq('video') & events.channel.eq('')).sum(),
               events.subtitle_count.gt(1).sum(), events.raw_record_hash.duplicated().sum()]
}))
display(events.loc[events.event_kind.ne('video'), ['source_row','event_kind','raw_title','raw_url']].head(12))
display(events.loc[events.details.ne(''), 'details'].value_counts().rename('events').to_frame())


In [ ]:
# Sanity checks cho cách tách URL và tính bảo toàn số lượng.
assert classify_url('https://www.youtube.com/watch?v=abcdefghijk&t=30') == ('video','abcdefghijk')
assert classify_url('https://youtu.be/abcdefghijk') == ('video','abcdefghijk')
assert classify_url('https://www.youtube.com/shorts/abcdefghijk') == ('video','abcdefghijk')
assert classify_url('https://www.youtube.com/post/example')[0] == 'community_post'
assert classify_url('https://example.com/watch?v=abcdefghijk')[0] == 'other_host'
assert classify_url('https://www.youtube.com/watch?v=bad')[0] == 'invalid_video_url'
assert classify_url(None)[0] == 'missing_url'
assert len(events) == len(raw)

views = events.loc[events.event_kind.eq('video') & ~events.is_ad].copy()
if views.empty:
    raise ValueError('Không có sự kiện video không đánh dấu quảng cáo để phân tích.')
# Giữ mọi sự kiện để dashboard; chỉ gom ID cho bảng video/thư viện tải.
ordered = views.sort_values(['time_utc','source_row'], na_position='first')
def last_nonempty(series):
    values = series[series.notna() & series.ne('')]
    return values.iloc[-1] if len(values) else ''

videos = ordered.groupby('video_id', as_index=False).agg(
    title=('title',last_nonempty), channel=('channel',last_nonempty),
    channel_url=('channel_url',last_nonempty), watch_count=('source_row','size'),
    first_seen=('time_utc','min'), last_seen=('time_utc','max'),
    title_variants=('title','nunique'), channel_variants=('channel','nunique'),
)
videos['url'] = 'https://www.youtube.com/watch?v=' + videos.video_id
videos['channel_key'] = videos.channel_url.where(videos.channel_url.ne(''), videos.channel)
videos.loc[videos.channel_key.eq(''), 'channel_key'] = '(missing channel)'
assert videos.video_id.is_unique
assert int(videos.watch_count.sum()) == len(views)
summary = pd.Series({
    'raw_events':len(events), 'video_events_including_ads':int(events.event_kind.eq('video').sum()),
    'ad_events_all_types':int(events.is_ad.sum()), 'eligible_watch_events':len(views),
    'unique_eligible_videos':len(videos), 'known_channel_keys':videos.loc[videos.channel_key.ne('(missing channel)'), 'channel_key'].nunique(),
    'videos_seen_more_than_once':int(videos.watch_count.gt(1).sum()),
    'earliest_event_utc':str(events.time_utc.min()), 'latest_event_utc':str(events.time_utc.max()),
}, name='value')
display(summary.to_frame())


## 3. Mức độ lặp, thời gian và kênh

Thống kê số lượt xuất hiện trong lịch sử, **không phải số lượt nghe trọn bài hoặc thời lượng nghe**. Không suy ra toàn bộ lịch sử tài khoản ngoài khoảng thời gian export. Kênh là thông tin từ subtitle của Takeout; tên có thể đổi hoặc thiếu, nên ưu tiên URL làm khóa.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))
bands = pd.cut(videos.watch_count, bins=[0,1,2,5,10,float('inf')], labels=['1','2','3–5','6–10','11+'])
bands.value_counts(sort=False).plot.bar(ax=axes[0], color='#4c78a8', rot=0)
axes[0].set(title='Repeat distribution (unique videos)', xlabel='Recorded watches', ylabel='Videos')
local_dates = views.time_local.dropna().dt.date
if len(local_dates):
    local_dates.value_counts().sort_index().plot(ax=axes[1], color='#f58518')
axes[1].set(title=f'Watch events by day ({LOCAL_TIMEZONE})', xlabel='Date', ylabel='Events')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'history_overview.png', dpi=160, bbox_inches='tight')
plt.show()
display(videos.watch_count.describe(percentiles=[.5,.9,.95,.99]).to_frame())

channels = videos.groupby('channel_key').agg(
    channel=('channel',last_nonempty), unique_videos=('video_id','size'), watch_events=('watch_count','sum')
).sort_values('unique_videos',ascending=False)
display(channels.head(20))
known_channels = channels.drop(index='(missing channel)', errors='ignore')
print('Top 20 known channels: share of unique videos =',
      round(known_channels.head(20).unique_videos.sum()/len(videos),3))
display(videos.nlargest(15,'watch_count')[['title','channel','watch_count','url']])


## 4. Đặc điểm tiêu đề trước khi nghĩ đến regex

Chuẩn hóa Unicode NFKC, viết thường và khoảng trắng; giữ nguyên tiêu đề gốc để duyệt. Các cờ Kana/Han/Hangul/Latin bên dưới là **hệ chữ**, không phải nhận diện ngôn ngữ: Han có thể xuất hiện trong nhiều ngôn ngữ và tiêu đề có thể trộn hệ chữ.

Tần suất token tính theo **số video chứa token**, không tính số lượt xem. Không dùng token phổ biến làm nhãn tự động; chúng có thể là tên nghệ sĩ, thương hiệu hoặc từ thông dụng.


In [ ]:
def normalize_title(value):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC',value).casefold()).strip()

videos['title_norm'] = videos.title.map(normalize_title)
videos['title_chars'] = videos.title_norm.str.len()
SCRIPT_PATTERNS = {
    'kana':r'[\u3040-\u30ff]', 'han':r'[\u3400-\u4dbf\u4e00-\u9fff]',
    'hangul':r'[\uac00-\ud7af\u1100-\u11ff]', 'latin':r'[a-zA-Z\u00c0-\u024f\u1e00-\u1eff]',
}
for name, pattern in SCRIPT_PATTERNS.items():
    videos['script_'+name] = videos.title.str.contains(pattern, regex=True, na=False)
videos['script_count'] = videos[['script_'+s for s in SCRIPT_PATTERNS]].sum(axis=1)
videos['short_title'] = videos.title_chars.lt(12)  # Giả thuyết khám phá, không phải ngưỡng phân loại.
script_summary = pd.DataFrame({
    'videos':{s:int(videos['script_'+s].sum()) for s in SCRIPT_PATTERNS},
})
script_summary['share'] = script_summary.videos / len(videos)
display(script_summary)
display(videos.title_chars.describe(percentiles=[.1,.25,.5,.75,.9]).to_frame())
print('Tiêu đề trộn >= 2 hệ chữ:', int(videos.script_count.ge(2).sum()))
print('Tiêu đề < 12 ký tự:', int(videos.short_title.sum()))

STOPWORDS = {'the','a','an','and','of','to','in','on','for','with','is','it','at','by'}
def title_tokens(title):
    return set(t for t in re.findall(r'[^\W_]+',title,flags=re.UNICODE)
               if len(t)>1 and not t.isdigit() and t not in STOPWORDS)
token_counts = Counter(t for title in videos.title_norm for t in title_tokens(title))
token_frequency = pd.DataFrame(sorted(token_counts.items(), key=lambda item: (-item[1], item[0]))[:60],columns=['token','unique_videos'])
display(token_frequency.head(40))
print('Tokenization theo khoảng trắng/ký tự: chưa tách từ tiếng Nhật/Trung/Thái; cần xem mẫu trực tiếp.')

same_titles = videos.groupby('title_norm').agg(video_ids=('video_id','size'),watch_events=('watch_count','sum')).query('video_ids > 1').sort_values('video_ids',ascending=False)
display(same_titles.head(15))
print('Cùng tiêu đề KHÔNG chứng minh cùng bài/bản thu; không gộp các ID khi tải.')


# Phân biệt hệ chữ trong phần tiêu đề chính và ký tự trang trí ở hashtag.
HASHTAG_PATTERN = r'(?<!\w)#[^\s#]+'
videos['title_body'] = videos.title_norm.str.replace(HASHTAG_PATTERN, '', regex=True).str.strip()
script_cols = ['script_'+s for s in ['kana','han','hangul']]
for name, pattern in SCRIPT_PATTERNS.items():
    videos['body_'+name+'_chars'] = videos.title_body.str.count(pattern)
videos['body_cjk_chars'] = videos[['body_'+s+'_chars' for s in ['kana','han','hangul']]].sum(axis=1)
videos['body_letter_chars'] = videos.title_body.map(lambda s: sum(c.isalpha() for c in s))
videos['body_cjk_share'] = videos.body_cjk_chars.div(videos.body_letter_chars.replace(0,float('nan'))).fillna(0)
videos['cjk_hashtag_only'] = videos[script_cols].any(axis=1) & videos.body_cjk_chars.eq(0)
# Ngưỡng để chọn mẫu, không phải mô hình nhận diện ngôn ngữ.
videos['cjk_substantive'] = videos.body_cjk_chars.ge(2) & videos.body_cjk_share.ge(.2)
display(pd.Series({'any_cjk_in_full_title':int(videos[script_cols].any(axis=1).sum()),
                  'cjk_in_hashtag_only':int(videos.cjk_hashtag_only.sum()),
                  'substantive_cjk_in_title_body':int(videos.cjk_substantive.sum())},name='videos').to_frame())
assert re.sub(HASHTAG_PATTERN,'','English title #fypシ').strip() == 'English title'
assert not re.search(SCRIPT_PATTERNS['kana'], re.sub(HASHTAG_PATTERN,'','English title #fypシ'))
assert re.search(SCRIPT_PATTERNS['kana'], re.sub(HASHTAG_PATTERN,'','カバー #music'))


## 5. Khám phá giả thuyết tín hiệu — chưa chấm điểm, chưa gán nhãn

Các nhóm dưới đây được khởi tạo từ phạm vi sản phẩm, **không phải quy tắc đã được xác nhận**. Đo độ phủ và giao nhau để chọn mẫu cần nghe/xem. Giữ `live`, `cover`, `mix`, `BGM` ở nhóm mơ hồ: livestream nói chuyện, hướng dẫn cover, game có BGM cũng có thể khớp.

Không có tín hiệu ⇒ thiếu bằng chứng, không đồng nghĩa `non_music`. Có nhiều tín hiệu ⇒ nhiều lần khớp, không phải xác suất nhạc. Nhạc instrumental không cần có hát; nhạc có lời không bị loại chỉ vì có giọng người.


In [ ]:
SIGNAL_PATTERNS = {
    'music_terms': r'(?<!\w)(?:ost|soundtrack|amv|mv|lyrics?|remix|instrumental|karaoke|nightcore)(?!\w)|music video|official audio|nhạc|âm nhạc|歌ってみた|オリジナル曲|뮤직',
    'ambiguous_terms': r'(?<!\w)(?:live|cover|mix|bgm|edit|version|opening|ending)(?!\w)|ライブ|カバー',
    'talk_context': r'(?<!\w)(?:reaction|review|tutorial|gameplay|walkthrough|podcast|interview|vlog)(?!\w)|hướng dẫn|phỏng vấn|bình luận',
}
for name, pattern in SIGNAL_PATTERNS.items():
    videos['signal_'+name] = videos.title_norm.str.contains(pattern, regex=True, na=False)
signal_columns = ['signal_'+s for s in SIGNAL_PATTERNS]
videos['signal_count'] = videos[signal_columns].sum(axis=1)
videos['signal_conflict'] = videos.signal_talk_context & (videos.signal_music_terms | videos.signal_ambiguous_terms)
videos['signal_none'] = videos.signal_count.eq(0)
display(pd.DataFrame([
    {'hypothesis':s,'unique_videos':int(videos['signal_'+s].sum()),
     'video_share':float(videos['signal_'+s].mean()),
     'watch_events':int(videos.loc[videos['signal_'+s],'watch_count'].sum())}
    for s in SIGNAL_PATTERNS
]))
display(videos[signal_columns].value_counts().rename('unique_videos').reset_index())
print({'no_title_signal':int(videos.signal_none.sum()), 'conflicting_signals':int(videos.signal_conflict.sum())})
# Một kênh có nhiều tín hiệu không đảm bảo mọi video của kênh là nhạc.
channel_signals = videos.groupby('channel_key').agg(
    unique_videos=('video_id','size'), music_term_share=('signal_music_terms','mean'),
    ambiguous_share=('signal_ambiguous_terms','mean'), talk_share=('signal_talk_context','mean'),
    no_signal_share=('signal_none','mean'),
).sort_values('unique_videos',ascending=False)
display(channel_signals.head(20))

# Kiểm tra cách chuẩn hóa và ranh giới từ, không kiểm định chất lượng phân loại.
assert normalize_title(' ＭＶ   TEST ') == 'mv test'
assert title_tokens('hello hello world') == {'hello', 'world'}
assert re.search(SCRIPT_PATTERNS['kana'], 'カバー')
assert not re.search(SCRIPT_PATTERNS['kana'], 'abc')
assert re.search(SIGNAL_PATTERNS['music_terms'], 'anime ost')
assert not re.search(SIGNAL_PATTERNS['music_terms'], 'lost in a forest')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
coverage = pd.Series({**{s: int(videos['signal_'+s].sum()) for s in SIGNAL_PATTERNS},
                      'no_signal': int(videos.signal_none.sum())})
coverage.plot.barh(ax=axes[0], color=['#54a24b','#f58518','#e45756','#bab0ac'])
axes[0].set(title='Exploratory title signals (groups can overlap)', xlabel='Unique videos')
videos.title_chars.plot.hist(bins=30, ax=axes[1], color='#4c78a8')
axes[1].set(title='Normalized title length', xlabel='Characters', ylabel='Unique videos')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'classification_evidence.png', dpi=160, bbox_inches='tight')
plt.show()


### 5.1. Bằng chứng từ kênh, thư viện nhạc và hành vi xem lại

Theo [YouTube Help — artist Topic channels](https://support.google.com/youtube/answer/7636475?hl=en), kênh nghệ sĩ tự tạo có tên “Artist Name - Topic”. [Art Track](https://support.google.com/youtube/answer/6007071?hl=en) là phiên bản tự tạo từ bản thu và ảnh album. Theo phạm vi đã thống nhất, **artist Topic là tín hiệu nhạc mạnh**. Takeout chỉ cho tên/URL kênh: cờ suffix bên dưới là dấu hiệu tên, chưa xác thực provenance qua YouTube.

Các tín hiệu mới được tách mức bằng chứng:
- **Nguồn mạnh:** tên kênh `- Topic`; ID khớp chính xác thư viện nhạc trong cùng Takeout. Không đổi chúng thành nhãn thủ công để đánh giá accuracy.
- **Hỗ trợ nội dung:** hậu tố VEVO, slowed/reverb/sped-up/lofi, nhạc cụ/biểu diễn, cấu trúc Artist – Title. VEVO/Official tự viết trong tên không chứng minh kênh đã được xác minh.
- **Liên kết nghệ sĩ:** tên kênh hoặc phần Artist trước dấu gạch trùng tên nghệ sĩ từ Topic trong export. Có thể tìm bản unofficial hoặc kênh nghệ sĩ; không whitelist mọi nội dung của kênh.
- **Hành vi:** số timestamp khác nhau, số ngày xem, khoảng cách lần đầu/cuối, tỷ trọng lượt tập trung trong một ngày. Đây là sự kiện lịch sử, không phải số lần nghe hết. Podcast, hướng dẫn và video yêu thích vẫn có thể xem lại.
- **Ngữ cảnh cần xem:** podcast/talk/game/tutorial, behind-the-scenes và hashtag clip ngắn. Không dùng hashtag Shorts để loại AMV/clip biểu diễn.

File thư viện là tùy chọn; thiếu file vẫn chạy EDA. Chỉ đối chiếu ID trong lịch sử, không tự thêm tất cả bài trong library vào danh sách tải. Các ngưỡng hành vi hiện là giả thuyết chọn mẫu, chưa phải điểm phân loại.

In [ ]:
# Tính hành vi từ sự kiện không đánh dấu ad, bỏ timestamp lặp CHỈ trong bảng đặc trưng hành vi.
valid_events = views.dropna(subset=['time_utc']).drop_duplicates(['video_id','time_utc']).copy()
valid_events['local_date'] = valid_events.time_local.dt.date
behavior = valid_events.groupby('video_id').agg(
    distinct_watch_times=('time_utc','size'), watch_days=('local_date','nunique'),
    behavior_first=('time_utc','min'), behavior_last=('time_utc','max'),
)
behavior['watch_span_days'] = (behavior.behavior_last-behavior.behavior_first).dt.total_seconds()/86400
behavior['max_watches_in_one_day'] = valid_events.groupby(['video_id','local_date']).size().groupby('video_id').max()
behavior['same_day_concentration'] = behavior.max_watches_in_one_day / behavior.distinct_watch_times
videos = videos.drop(columns=list(behavior.columns),errors='ignore')
videos = videos.merge(behavior.drop(columns=['behavior_first','behavior_last']),on='video_id',how='left',validate='one_to_one')
for col in ['distinct_watch_times','watch_days','max_watches_in_one_day']:
    videos[col] = videos[col].fillna(0).astype(int)
videos['signal_repeat_days'] = videos.distinct_watch_times.ge(3) & videos.watch_days.ge(3)
videos['signal_repeat_span'] = videos.signal_repeat_days & videos.watch_span_days.ge(7)
TOPIC_SUFFIX = r'\s+[-–—]\s*topic\s*$'
videos['channel_norm'] = videos.channel.map(normalize_title)
videos['signal_topic_channel'] = videos.channel_norm.str.contains(TOPIC_SUFFIX,regex=True)
videos['signal_vevo_channel'] = videos.channel_norm.str.contains(r'vevo$',regex=True)

# Thư viện nhạc: tìm trong cùng subtree export, tránh lấy nhầm export khác.
library_root = source.parent.parent
library_paths = sorted(library_root.rglob('music library songs.csv'))
if len(library_paths)>1:
    raise ValueError('Có nhiều file music library songs.csv; chọn đúng subtree Takeout trước khi chạy.')
library_ids=set()
library_rows=0
library_hash=None
if library_paths:
    library_source=library_paths[0]
    library_hash=sha256(library_source.read_bytes()).hexdigest()
    library=pd.read_csv(library_source,dtype=str,keep_default_na=False)
    if 'Video ID' not in library:
        raise ValueError('File music library songs.csv thiếu cột Video ID.')
    library_rows=len(library)
    library_ids={s.strip() for s in library['Video ID'] if VIDEO_ID.fullmatch(s.strip())}
videos['signal_music_library'] = videos.video_id.isin(library_ids)

EXTRA_PATTERNS = {
    'versions_genres': r'(?<!\w)(?:slowed|reverb(?:ed)?|sped[ -]?up|speed[ -]?up|lo[ -]?fi|city[ -]?pop|synthwave|vaporwave)(?!\w)',
    'instruments_performance': r'(?<!\w)(?:piano|violin|erhu|symphony|orchestra|orchestral|guitar|saxophone|cello|fingerstyle|oboe|concerto)(?!\w)|ピアノ|弾いてみた|演奏|피아노|钢琴|鋼琴|二胡',
    'song_multilingual': r'歌ってみた|オリジナル曲|カバー|커버|노래|翻唱|纯音乐|純音樂|nhạc không lời|bản phối',
    'spoken_context': r'(?<!\w)(?:podcast|interview|reaction|review|gameplay|walkthrough|vlog|commentary)(?!\w)|behind the scenes|making of|phỏng vấn|bình luận|tâm sự|chuyện trò',
    'tutorial_context': r'(?<!\w)(?:tutorial|lesson|how to)(?!\w)|hướng dẫn|cách chơi|cách đánh|教學|教学|강좌',
    'shortform_context': r'(?<!\w)#(?:shorts|shortvideo|ytshorts|fyp|foryou)(?!\w)',
    'artist_title_shape': r'^.{2,70}?\s[-–—]\s\S',
}
for name,pattern in EXTRA_PATTERNS.items():
    videos['signal_'+name]=videos.title_norm.str.contains(pattern,regex=True)
artists=set(videos.loc[videos.signal_topic_channel,'channel_norm'].str.replace(TOPIC_SUFFIX,'',regex=True))
artists={s for s in artists if len(s)>=2}
artist_prefix=videos.title_norm.str.split(r'\s[-–—]\s',n=1,regex=True).str[0]
videos['signal_artist_link']=(videos.channel_norm.isin(artists) |
    (videos.signal_artist_title_shape & artist_prefix.isin(artists))) & ~videos.signal_topic_channel
videos['strong_music_evidence']=videos.signal_topic_channel | videos.signal_music_library
support_columns=['signal_music_terms','signal_ambiguous_terms','signal_vevo_channel',
                 'signal_versions_genres','signal_instruments_performance','signal_song_multilingual','signal_artist_link']
videos['content_music_hint']=videos[support_columns].any(axis=1)
videos['review_context']=videos.signal_spoken_context | videos.signal_tutorial_context
videos['content_conflict']=videos.content_music_hint & videos.review_context
# Giữ nguyên signal_none để so sánh baseline title-only, đặt tên riêng cho khoảng trống mới.
videos['no_music_evidence_yet']=~(videos.strong_music_evidence | videos.content_music_hint | videos.signal_repeat_days)
videos['review_bucket']='insufficient_music_evidence'
videos.loc[videos.signal_repeat_days,'review_bucket']='repeat_behavior_only'
videos.loc[videos.content_music_hint,'review_bucket']='content_hint_review'
videos.loc[videos.review_context,'review_bucket']='context_review'
videos.loc[videos.content_conflict,'review_bucket']='music_context_conflict'
videos.loc[videos.strong_music_evidence,'review_bucket']='strong_topic_or_library'
# Bucket dùng điều phối review, không ghi vào manual_label, không tự loại video.
evidence_labels={
    'signal_topic_channel':'Topic suffix', 'signal_music_library':'music library ID',
    'signal_vevo_channel':'VEVO suffix', 'signal_music_terms':'music title terms',
    'signal_ambiguous_terms':'ambiguous title terms','signal_versions_genres':'version/genre',
    'signal_instruments_performance':'instrument/performance','signal_song_multilingual':'song vocabulary',
    'signal_artist_link':'artist link','signal_artist_title_shape':'Artist - Title shape (weak)',
    'signal_repeat_days':'repeat on >=3 days','signal_repeat_span':'repeat spans >=7 days',
    'signal_spoken_context':'talk/game/BTS context','signal_tutorial_context':'tutorial context',
    'signal_shortform_context':'shortform hashtag',
}
videos['evidence']=videos.apply(lambda row:'; '.join(label for col,label in evidence_labels.items() if row[col]) or 'no observed evidence',axis=1)
feature_coverage=pd.DataFrame([
    {'feature':col,'videos':int(videos[col].sum()),
     'outside_baseline_title_signals':int((videos[col]&videos.signal_none).sum()),
     'outside_topic_library':int((videos[col]&~videos.strong_music_evidence).sum()),
     'with_review_context':int((videos[col]&videos.review_context).sum())}
    for col in evidence_labels
])
display(feature_coverage)
display(videos.review_bucket.value_counts().rename('videos').to_frame())
print({'library_rows':library_rows,'valid_library_ids':len(library_ids),
       'library_IDs_in_history':int(videos.signal_music_library.sum()),
       'topic_videos':int(videos.signal_topic_channel.sum()),
       'topic_missed_by_title_signals':int((videos.signal_topic_channel&videos.signal_none).sum())})
assert re.search(TOPIC_SUFFIX,'artist - topic')
assert not re.search(TOPIC_SUFFIX,'a topic discussion')
assert re.search(EXTRA_PATTERNS['versions_genres'],normalize_title('𝚂𝚕𝚘𝚠𝚎𝚍'))
assert not re.search(EXTRA_PATTERNS['versions_genres'],'delivery service')
assert not videos.loc[videos.distinct_watch_times.lt(3),'signal_repeat_days'].any()

In [ ]:
# Topic/library chỉ là nhóm tham chiếu dương có thiên lệch catalog, KHÔNG phải ground truth đầy đủ.
repeat_band=pd.cut(videos.distinct_watch_times,bins=[-1,1,2,4,9,float('inf')],labels=['0–1','2','3–4','5–9','10+'])
repeat_analysis=videos.assign(repeat_band=repeat_band).groupby('repeat_band',observed=False).agg(
    videos=('video_id','size'), topic_or_library=('strong_music_evidence','sum'),
    content_hints=('content_music_hint','sum'), talk_or_tutorial=('review_context','sum'),
    median_watch_days=('watch_days','median'),
)
repeat_analysis['topic_or_library_share']=repeat_analysis.topic_or_library/repeat_analysis.videos.replace(0,float('nan'))
display(repeat_analysis)
print('Tỷ lệ Topic/library theo lượt xem là tương quan trong export này; KHÔNG là P(music | repeats).')
channel_profile=videos.groupby('channel_key').agg(
    channel=('channel',last_nonempty),unique_videos=('video_id','size'),
    watched_on_3_days=('signal_repeat_days','sum'),strong_evidence=('strong_music_evidence','sum'),
    content_hint_share=('content_music_hint','mean'),review_context_share=('review_context','mean'),
).sort_values('watched_on_3_days',ascending=False)
display(channel_profile.head(15))
fig,axes=plt.subplots(1,2,figsize=(14,4))
repeat_analysis.topic_or_library_share.mul(100).plot.bar(ax=axes[0],rot=0,color='#54a24b')
axes[0].set(title='Topic/library reference share by repeats',ylabel='% of videos in each band',xlabel='Distinct watch timestamps')
feature_coverage.set_index('feature').outside_baseline_title_signals.sort_values().plot.barh(ax=axes[1],color='#4c78a8')
axes[1].set(title='New signals outside title-only baseline',xlabel='Unique videos (overlap possible)')
plt.tight_layout()
fig.savefig(OUTPUT_DIR/'additional_evidence.png',dpi=160,bbox_inches='tight')
plt.show()

In [ ]:
# Bảng sample nhỏ gọn, cuộn ngang, có lý do khớp; không tự tải ảnh/metadata ngoài máy.
def preview_table(frame,n=10):
    sample=frame.head(n)
    if sample.empty:
        print('(Không có video trong nhóm này)')
        return
    cols=['video_id','title','channel','watch_count','watch_days','review_bucket','evidence']
    if 'sampling_stratum' in sample.columns:
        cols.insert(0,'sampling_stratum')
    table=sample[cols].copy()
    for col in table.columns:
        table[col]=table[col].map(lambda s:html.escape(str(s)))
    table['title']=[f'<a href="{html.escape(u,quote=True)}" target="_blank" rel="noopener noreferrer">{t}</a>'
                    for u,t in zip(sample.url,table.title)]
    display(HTML('<div style="overflow-x:auto;max-height:520px">'+table.to_html(index=False,escape=False)+'</div>'))

def channel_diverse_sample(frame,n,seed,max_per_channel=2):
    counts=Counter()
    selected=[]
    for idx,row in frame.sample(frac=1,random_state=seed).iterrows():
        key=row.channel_key if row.channel_key!='(missing channel)' else row.video_id
        if counts[key]>=max_per_channel:
            continue
        selected.append(idx)
        counts[key]+=1
        if len(selected)>=n:
            break
    return frame.loc[selected].copy()

# Cũng giữ random evaluation ngoài các bảng khám phá để tránh chỉnh quy tắc theo holdout.
evaluation_ids=set(videos.sample(n=min(RANDOM_SAMPLE_SIZE,len(videos)),random_state=SEED).video_id)
preview_pool=videos.loc[~videos.video_id.isin(evaluation_ids)].copy()
groups={
    'Topic/library — title-only đã bỏ sót':preview_pool.strong_music_evidence & preview_pool.signal_none,
    'Nhạc cụ/phiên bản/nghệ sĩ ngoài Topic':~preview_pool.strong_music_evidence & (preview_pool.signal_instruments_performance | preview_pool.signal_versions_genres | preview_pool.signal_artist_link),
    'Xem nhiều ngày nhưng có talk/tutorial/BTS':preview_pool.signal_repeat_days & preview_pool.review_context,
    'Xem nhiều ngày, chưa có tín hiệu nội dung':preview_pool.signal_repeat_days & ~preview_pool.content_music_hint & ~preview_pool.strong_music_evidence,
    'Tiêu đề ngắn, chưa có bằng chứng nhạc':preview_pool.short_title & preview_pool.no_music_evidence_yet,
    'Kana/Han/Hangul trong phần tiêu đề chính':preview_pool.cjk_substantive & ~preview_pool.strong_music_evidence,
    'Ký tự Kana/Han/Hangul chỉ ở hashtag':preview_pool.cjk_hashtag_only,
}
for i,(label,mask) in enumerate(groups.items()):
    print(label,'—',int(mask.sum()),'videos trong discovery pool')
    preview_table(channel_diverse_sample(preview_pool.loc[mask],8,SEED+i))

## 6. Tạo bộ mẫu gán nhãn trước khi thiết kế trọng số

- **Random / evaluation:** lấy ngẫu nhiên đều theo video ID, không theo lượt xem; giữ riêng để đánh giá quy tắc đã chốt. Không xem nhãn tập này khi tinh chỉnh quy tắc. Với mẫu nhỏ, phải báo số lượng và độ bất định; không coi độ phủ từ khóa là accuracy.
- **Discovery:** ưu tiên ca không có tín hiệu, mâu thuẫn, mơ hồ, tiêu đề ngắn, hệ chữ khác và video lặp nhiều. Không giao với tập random; một video chỉ xuất hiện một lần. Mẫu này có thiên lệch chủ đích, không dùng để ước lượng tỉ lệ trên toàn lịch sử.
- Nhãn hợp lệ: `music`, `non_music`, `uncertain`, `unavailable`. Giữ trống khi chưa review; ghi lý do vào `notes`. Không đoán nhãn từ title khi thực sự cần mở video.

Cỡ mẫu hiện tại là điểm bắt đầu cho pilot, chưa phải chứng minh mức accuracy mục tiêu. Chia train/test theo kênh ở giai đoạn sau nếu muốn đo khả năng khái quát sang kênh mới.


**Revision signals-v2:** sample giữ riêng trong thư mục version mới, không sửa mẫu/nhãn cũ. Random vẫn cùng 300 ID nếu source/seed/cỡ mẫu không đổi. Discovery giới hạn tối đa 2 video/kênh/nhóm và 5 video/kênh toàn bộ; nhóm thiếu ứng viên có thể nhỏ hơn 25. Có nhóm Topic/library riêng, còn nhóm “unresolved” loại các video đã có bằng chứng mạnh.

Nhãn discovery cũ được mang sang theo ID khi xuất mẫu mới nếu hai mẫu giao nhau. Không sao chép nhãn random vào discovery. Ảnh preview ở phần trên là bảng khám phá, không đồng nghĩa các hàng đó là nhạc.


In [ ]:
random_review=videos.sample(n=min(RANDOM_SAMPLE_SIZE,len(videos)),random_state=SEED).copy()
random_review['sampling_stratum']='uniform_random_video'
random_review['cohort']='random_evaluation'
pool=videos.loc[~videos.video_id.isin(random_review.video_id)].copy()
strata={
    'topic_library_title_miss':pool.strong_music_evidence & pool.signal_none,
    'music_context_conflict':pool.content_conflict & ~pool.strong_music_evidence,
    'repeat_with_spoken_context':pool.signal_repeat_days & pool.signal_spoken_context & ~pool.strong_music_evidence,
    'repeat_only_unresolved':pool.signal_repeat_days & ~pool.content_music_hint & ~pool.strong_music_evidence,
    'versions_outside_topic':pool.signal_versions_genres & ~pool.strong_music_evidence,
    'instruments_outside_topic':pool.signal_instruments_performance & ~pool.strong_music_evidence,
    'vevo_artist_link':(pool.signal_vevo_channel | pool.signal_artist_link) & ~pool.strong_music_evidence,
    'short_title_unresolved':pool.short_title & pool.no_music_evidence_yet,
    'cjk_body_unresolved':pool.cjk_substantive & ~pool.strong_music_evidence,
    'hashtag_script_only':pool.cjk_hashtag_only,
    'no_music_evidence':pool.no_music_evidence_yet,
    'missing_channel':pool.channel.eq(''),
}
chosen=[]
used=set()
global_channel_counts=Counter()
sampling_audit=[]
for i,(name,mask) in enumerate(strata.items()):
    candidates=pool.loc[mask & ~pool.video_id.isin(used)]
    candidate_keys=candidates.channel_key.where(candidates.channel_key.ne('(missing channel)'),candidates.video_id)
    candidates=candidates.loc[candidate_keys.map(global_channel_counts).fillna(0).lt(MAX_PER_CHANNEL_DISCOVERY)]
    shuffled=candidates.sample(frac=1,random_state=SEED+i)
    selected=[]
    local_counts=Counter()
    for idx,row in shuffled.iterrows():
        key=row.channel_key if row.channel_key!='(missing channel)' else row.video_id
        if local_counts[key]>=MAX_PER_CHANNEL_PER_STRATUM or global_channel_counts[key]>=MAX_PER_CHANNEL_DISCOVERY:
            continue
        selected.append(idx)
        local_counts[key]+=1
        global_channel_counts[key]+=1
        if len(selected)>=DISCOVERY_SAMPLE_PER_STRATUM:
            break
    sample=pool.loc[selected].assign(sampling_stratum=name,cohort='discovery')
    chosen.append(sample)
    used.update(sample.video_id)
    sampling_audit.append({'stratum':name,'eligible_before_overlap_caps':int(mask.sum()),'sampled':len(sample),'sample_channels':len(local_counts)})
discovery_review=pd.concat(chosen,ignore_index=True)
assert not set(random_review.video_id)&set(discovery_review.video_id)
assert discovery_review.video_id.is_unique
assert max(global_channel_counts.values(),default=0)<=MAX_PER_CHANNEL_DISCOVERY

def valid_annotations(frame,source_name):
    if not {'video_id','manual_label','notes'}<=set(frame.columns) or frame.video_id.duplicated().any():
        raise ValueError(f'Review thiếu cột hoặc trùng ID: {source_name}')
    frame=frame.copy()
    frame['manual_label']=frame.manual_label.str.strip().str.casefold()
    if not set(frame.manual_label)<={'','music','non_music','uncertain','unavailable'}:
        raise ValueError(f'Review có nhãn không hợp lệ: {source_name}')
    return frame[['video_id','manual_label','notes']]

review_columns=['video_id','title','channel','url','watch_count','watch_days','review_bucket','evidence',
                'cohort','sampling_stratum',*signal_columns,'signal_topic_channel','signal_music_library',
                'signal_repeat_days','manual_label','notes']
for name,sample in [('random_review.csv',random_review),('discovery_review.csv',discovery_review)]:
    path=OUTPUT_DIR/name
    if path.exists():
        existing=pd.read_csv(path,keep_default_na=False)
        valid_annotations(existing,name)
        if set(existing.video_id)!=set(sample.video_id):
            raise ValueError(f'Mẫu đã lưu khác cấu hình: {path}. Đổi EDA_VERSION để giữ nguyên nhãn cũ.')
        print(f'Giữ nguyên {name}: {len(existing)} videos')
        continue
    legacy_path=LEGACY_OUTPUT_DIR/name
    # Chỉ mang nhãn/notes cùng cohort; mọi feature dùng giá trị mới.
    if legacy_path.exists():
        annotation=valid_annotations(pd.read_csv(legacy_path,keep_default_na=False),str(legacy_path))
        sample=sample.merge(annotation,on='video_id',how='left',validate='one_to_one')
        sample[['manual_label','notes']]=sample[['manual_label','notes']].fillna('')
    else:
        sample=sample.assign(manual_label='',notes='')
    sample[review_columns].to_csv(path,index=False)
    print(f'Đã tạo {name}: {len(sample)} videos')
sampling_audit=pd.DataFrame(sampling_audit)
display(sampling_audit)
print('Review folder:',OUTPUT_DIR)
print('Mỗi nhóm discovery mới: tối đa 2 ví dụ đầu; không phải nhãn ground truth.')
preview_table(discovery_review.groupby('sampling_stratum',sort=False).head(2),n=30)


In [ ]:
# Audit sample discovery cũ bằng feature mới; không đọc nhãn/tiêu đề random holdout.
legacy_discovery_path=LEGACY_OUTPUT_DIR/'discovery_review.csv'
if legacy_discovery_path.exists():
    old_sample=pd.read_csv(legacy_discovery_path,keep_default_na=False)
    audit=old_sample[['video_id','sampling_stratum']].merge(videos,on='video_id',validate='one_to_one')
    sample_revision_audit=audit.groupby('sampling_stratum').agg(
        sample_size=('video_id','size'),topic_or_library=('strong_music_evidence','sum'),
        new_content_hint=('content_music_hint','sum'),repeat_days=('signal_repeat_days','sum'),
        hashtag_cjk_only=('cjk_hashtag_only','sum'),
    )
    display(sample_revision_audit)
    print('Các hàng từng ở no_signal/short_title nay có bằng chứng mạnh:')
    affected=audit.loc[audit.sampling_stratum.isin(['no_signal','short_title']) & audit.strong_music_evidence]
    preview_table(affected,12)
    sample_revision_audit.to_csv(OUTPUT_DIR/'legacy_discovery_audit.csv')
else:
    print('Không có discovery sample cũ để so sánh.')

## 7. Đọc nhãn khám phá sau khi người dùng review

Chạy lại cell này sau khi điền `discovery_review.csv`. Chỉ kiểm tra nhãn khám phá ở notebook 01; nhãn random giữ cho notebook đánh giá sau. Chưa có nhãn thì chưa có cơ sở chọn trọng số, ngưỡng hoặc công bố accuracy.


In [ ]:
reviewed=pd.read_csv(OUTPUT_DIR/'discovery_review.csv',keep_default_na=False)
required={'video_id','manual_label','notes'}
if not required <= set(reviewed.columns):
    raise ValueError(f'Thiếu cột review: {required-set(reviewed.columns)}')
if reviewed.video_id.duplicated().any() or set(reviewed.video_id) != set(discovery_review.video_id):
    raise ValueError('ID trong discovery_review.csv bị trùng, mất hoặc thêm ngoài mẫu; chỉ sửa nhãn và notes.')
reviewed['manual_label']=reviewed.manual_label.str.strip().str.casefold()
allowed={'','music','non_music','uncertain','unavailable'}
invalid=set(reviewed.manual_label)-allowed
if invalid:
    raise ValueError(f'Nhãn không hợp lệ: {sorted(invalid)}')
labeled=reviewed.loc[reviewed.manual_label.ne(''),['video_id','manual_label','notes']].merge(videos,on='video_id',validate='one_to_one')
if labeled.empty:
    print('Chưa có nhãn thủ công. Độ phủ tín hiệu ở trên không phải precision/recall.')
else:
    display(labeled.manual_label.value_counts().rename('reviewed_videos').to_frame())
    display(pd.crosstab(labeled.manual_label,labeled.review_bucket))
    print('Đây là mẫu khám phá có chủ đích; không ngoại suy các tỉ lệ này cho toàn lịch sử.')
    display(labeled.loc[(labeled.manual_label.eq('music') & labeled.signal_none) |
                        (labeled.manual_label.eq('non_music') & labeled.signal_music_terms),
                       ['video_id','title','manual_label','notes','url']].head(20))


## 8. Kết quả mang sang giai đoạn thiết kế

Dùng bằng chứng theo thứ tự:

| Quan sát | Quyết định cần kiểm chứng tiếp |
| --- | --- |
| Nhiều video nhạc đã gán nhãn không có từ khóa | Bổ sung metadata/tín hiệu kênh; không biến “không khớp” thành loại bỏ |
| Từ khóa khớp nhiều video không phải nhạc | Thêm ngữ cảnh và giữ nhóm mâu thuẫn để duyệt; chưa vội tăng trọng số |
| Kênh có nội dung trộn | Không whitelist toàn bộ kênh chỉ dựa trên vài video |
| Nhiều tiêu đề ít thông tin hoặc đa hệ chữ | Khảo sát riêng; cân nhắc metadata enrichment trước audio |
| Số lượt xem tập trung vào ít ID | Phân loại/tải theo video ID, dashboard theo sự kiện |
| URL thiếu hoặc video không còn truy cập được | Giữ trạng thái không xác định/không truy cập được, không suy ra non-music |

**Bước tiếp theo:** review mẫu discovery, ghi ví dụ phản chứng, rồi thiết kế bộ tín hiệu ở notebook 02. Chỉ quyết định có cần phân tích audio khi đo được khối lượng và bản chất của ca chưa rõ. Bản gốc tải xuống vẫn giữ audio stream, không chuyển mã.

Các số liệu tự động phía dưới là mô tả dữ liệu, không kết luận video nào là nhạc.


**Sau revision v2:** kiểm tra Topic/library trước; tách bằng chứng nội dung khỏi hành vi xem lặp và dấu hiệu trình bày. Ưu tiên review các tổ hợp nhạc cụ + tutorial, repeat + podcast/BTS, và Artist–Title không có bằng chứng khác. Không tăng trọng số chỉ vì một tín hiệu phổ biến trong sample có thiên lệch. `cjk_substantive` chỉ giúp chọn mẫu đa hệ chữ, không phải tín hiệu nhạc.


In [ ]:
report={
    'source_sha256':source_hash,'eda_version':EDA_VERSION,'library_sha256':library_hash,'seed':SEED,'local_timezone':LOCAL_TIMEZONE,
    'summary':{str(k):int(v) if hasattr(v,'item') else v for k,v in summary.items()},
    'signal_hypotheses':SIGNAL_PATTERNS,
    'signal_coverage':{s:int(videos['signal_'+s].sum()) for s in SIGNAL_PATTERNS},
    'no_title_signal':int(videos.signal_none.sum()),
    'topic_videos':int(videos.signal_topic_channel.sum()),
    'topic_title_misses':int((videos.signal_topic_channel & videos.signal_none).sum()),
    'library_matches':int(videos.signal_music_library.sum()),
    'cjk_hashtag_only':int(videos.cjk_hashtag_only.sum()),
    'review_buckets':{k:int(v) for k,v in videos.review_bucket.value_counts().items()},
    'additional_features':feature_coverage.to_dict(orient='records'),
    'conflicting_title_signals':int(videos.signal_conflict.sum()),
    'random_sample_videos':len(random_review),'discovery_sample_videos':len(discovery_review),
    'manual_discovery_labels':len(labeled),
    'classification_status':'not_implemented; signals are exploratory hypotheses',
    'limitations':['JSON input only','No observed watch duration','Script markers are not language labels',
                   'No availability check or metadata enrichment','No accuracy claim without independent labels'],
}
(OUTPUT_DIR/'eda_summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
schema.to_csv(OUTPUT_DIR/'schema.csv',index=False)
videos.to_csv(OUTPUT_DIR/'video_features.csv',index=False)
channels.to_csv(OUTPUT_DIR/'channel_summary.csv')
token_frequency.to_csv(OUTPUT_DIR/'title_token_frequency.csv',index=False)
print('Video thiếu tín hiệu tiêu đề:',report['no_title_signal'],'/',len(videos))
print('Video có tín hiệu mâu thuẫn:',report['conflicting_title_signals'])
print('Nhãn discovery đã điền:',len(labeled),'/',len(discovery_review))
print('Đã lưu summary, bảng đặc trưng, tần suất token, bảng kênh, biểu đồ và hai bộ mẫu tại:',OUTPUT_DIR)

feature_coverage.to_csv(OUTPUT_DIR/'feature_coverage.csv',index=False)
repeat_analysis.to_csv(OUTPUT_DIR/'repeat_analysis.csv')
channel_profile.to_csv(OUTPUT_DIR/'channel_evidence.csv')
sampling_audit.to_csv(OUTPUT_DIR/'sampling_audit.csv',index=False)
